# Scraping Zara.com
Collecte des données produits via Selenium (Firefox headless) + regex sur le HTML rendu.

**Variables collectées :** nom, url, categorie, couleur, matiere, collection, pays, prix, promo, reference

## 0. Installation des dépendances

In [ ]:
# À lancer une seule fois si les packages ne sont pas installés
# !pip install selenium webdriver-manager beautifulsoup4 pandas

## 1. Imports & Configuration

In [14]:
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import WebDriverException
from webdriver_manager.firefox import GeckoDriverManager
import re
import time
import pandas as pd
from bs4 import BeautifulSoup #Pour extraire et manipuler des données HTML


In [17]:
# Colonnes du DataFrame final
VARIABLES = ["nom", "url", "couleur", "matiere", "prix"]

df_zara = pd.DataFrame(columns=VARIABLES)
df_zara.head()

,nom,url,couleur,matiere,prix


In [4]:
# URLs de départ — pages liste de produits par collection
# (changer l'URL pour cibler une autre catégorie ou promotion)
URLS_LISTES = {
    "femme":   "https://www.zara.com/fr/fr/femme-nouveau-l1180.html",
    "homme":   "https://www.zara.com/fr/fr/homme-nouveau-l711.html",
    "fille": "https://www.zara.com/fr/fr/enfants-fille-nouveau-l391.html",
    "garçon": "https://www.zara.com/fr/fr/enfants-garcon-nouveau-l228.html"
}

PAYS = "France"

## 2. Lancement du navigateur (Firefox headless)

In [5]:
# headless=True = pas de fenêtre visible (recommandé pour du scraping en batch)
# Mettre headless=False si tu veux voir le navigateur s'ouvrir
HEADLESS = True

options = Options()
if HEADLESS:
    options.add_argument("--headless")

driver = webdriver.Firefox(
    service=Service(GeckoDriverManager().install()),
    options=options
)
print("Navigateur lancé.")

Navigateur lancé.


## 3. Extraction des URLs produits depuis une page liste

In [6]:
def get_product_urls(driver, listing_url, pause=8):
    """
    Charge une page liste Zara et extrait toutes les URLs produit.
    Les URLs produit Zara contiennent '-p' suivi de chiffres.
    """
    driver.get(listing_url)
    time.sleep(pause)

    # Scroll pour charger les produits en lazy-loading
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 2);")
    time.sleep(2)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

    links = driver.find_elements(By.TAG_NAME, "a")
    urls = set()
    for link in links:
        href = link.get_attribute("href")
        if href and "zara.com/fr/fr/" in href and re.search(r'-p\d+\.html', href):
            # Nettoyer les paramètres d'URL pour garder l'URL canonique
            clean = re.sub(r'\?.*$', '', href)
            urls.add(clean)

    return list(urls)

# Test sur femme
urls_femme = get_product_urls(driver, URLS_LISTES["femme"])
print(f"Femme : {len(urls_femme)} URLs trouvées")
urls_femme[:3]

Femme : 33 URLs trouvées


['https://www.zara.com/fr/fr/jupe-asymetrique-fluide-p01165021.html',
 'https://www.zara.com/fr/fr/short-mini-denim-trf-etoiles-p06929004.html',
 'https://www.zara.com/fr/fr/robe-midi-fluide-p04772370.html']

## 4. Dictionnaire de couleurs (hex → nom)

## 5. Fonction de scraping d'un produit

In [32]:
def scrape_produit(driver, url, pause=6):
    """
    Scrape les informations d'une page produit Zara.
    
    Retourne un dict avec :
      nom, url, categorie, couleur, matiere, pays, prix, promo, reference
    """
    try:
        driver.get(url)
        time.sleep(pause)
        html = driver.page_source
    except WebDriverException as e:
        print(f"  [ERREUR WebDriver] {url} : {e}")
        return None
    soup = BeautifulSoup(html, "html.parser")

    # ── NOM ────────────────────────────────────────────────────────────────
    match = re.search(r'"name":"([^"]{3,100})"', html)
    nom = match.group(1) if match else None

    # ── COULEUR ───────────────────────────────────────────────────────────

    color_tag = soup.select_one(
        '[data-qa-qualifier="product-detail-info-color"]'
    )

    if color_tag:
        button = color_tag.find("button")
        if button:
            button.extract()

        couleur = color_tag.get_text(" ", strip=True).strip('"').strip()
    else:
        couleur = None

    # ── MATIÈRE PRINCIPALE ────────────────────────────────────────────────
    match = re.search(
        r'"material":"([^"]+)","percentage":"?([0-9]+)',
        html,
        flags=re.IGNORECASE
    )
    if match:
        matiere_principale = f"{match.group(2)}% {match.group(1)}"
    else:
        matiere_principale = None

    # ── PRIX ──────────────────────────────────────────────────────────────
    price_tag = soup.select_one(".money-amount__main")
    if price_tag:
        prix = price_tag.get_text(" ", strip=True)
    else:
        prix = None

    return {
        "nom":        nom,
        "url":        url,
        "couleur":    couleur,
        "matiere":    matiere_principale,
        "prix":       prix,
    }



In [33]:
# ── TEST sur une URL ─────────────────────────────────────────────────────
if urls_femme:
    test = scrape_produit(driver, urls_femme[0])
    print(test)

{'nom': 'JUPE ASYMÉTRIQUE FLUIDE', 'url': 'https://www.zara.com/fr/fr/jupe-asymetrique-fluide-p01165021.html', 'couleur': 'Sable |', 'matiere': '76% polyester', 'prix': '29,95 EUR'}


## 6. Scraping complet — toutes collections

In [39]:
def scrape_collection(driver, listing_url, pause=6, max=50):
    """
    Récupère toutes les URLs d'une page liste puis scrape chaque produit.
    max_produits : limite le nombre de produits (None = tous).
    """
    urls = get_product_urls(driver, listing_url)
    print(f"  {len(urls)} URLs trouvées")
    urls = urls[:max]

    rows = []
    for i, url in enumerate(urls, 1):
        print(f"  [{i}/{len(urls)}] {url}")
        row = scrape_produit(driver, url,pause=pause)
        if row:
            rows.append(row)

    df = pd.DataFrame(rows, columns=VARIABLES)
    print(f"  → {len(df)} produits récupérés")
    return df

In [ ]:
# ── FEMME ────────────────────────────────────────────────────────────────
# max_produits=20 pour un test rapide ; mettre None pour tout scraper
df_femme = scrape_collection(driver, URLS_LISTES["femme"],max=None)
df_femme.head()

  33 URLs trouvées
  [1/33] https://www.zara.com/fr/fr/jupe-asymetrique-fluide-p01165021.html
  [2/33] https://www.zara.com/fr/fr/short-mini-denim-trf-etoiles-p06929004.html
  [3/33] https://www.zara.com/fr/fr/robe-midi-fluide-p04772370.html
  [4/33] https://www.zara.com/fr/fr/chemise-fluide-a-manches-courtes-p02906210.html
  [5/33] https://www.zara.com/fr/fr/sandales-plates-en-daim-a-boucle-p13646710.html
  [6/33] https://www.zara.com/fr/fr/jean-trf-loose-folded-taille-basse-a-rayures-p08727114.html
  [7/33] https://www.zara.com/fr/fr/chemise-cintree-froncee-p04661097.html
  [8/33] https://www.zara.com/fr/fr/short-fluide-a-boutons-p06929421.html
  [9/33] https://www.zara.com/fr/fr/short-en-denim-trf-dechire-taille-moyenne-p05252006.html
  [10/33] https://www.zara.com/fr/fr/combinaison-courte-en-lyocell-p06929485.html
  [11/33] https://www.zara.com/fr/fr/pull-asymetrique-en-maille-p09598016.html
  [12/33] https://www.zara.com/fr/fr/pantalon-droit-a-rayures-p02768116.html
  [13/33] http

,nom,url,couleur,matiere,prix
0,JUPE ASYMÉTRIQUE FLUIDE,https://www.zara.com/fr/fr/jupe-asymetrique-fl...,Sable |,76% polyester,"29,95 EUR"
1,SHORT MINI DENIM TRF ÉTOILES,https://www.zara.com/fr/fr/short-mini-denim-tr...,Bleu |,100% coton,"29,95 EUR"
2,ROBE MI-LONGUE FLUIDE,https://www.zara.com/fr/fr/robe-midi-fluide-p0...,Mauve poudré |,88% viscose,"25,99 EUR"
3,CHEMISE FLUIDE À MANCHES COURTES,https://www.zara.com/fr/fr/chemise-fluide-a-ma...,Mauve clair |,83% viscose,"25,95 EUR"
4,SANDALES PLATES EN DAIM À BOUCLE,https://www.zara.com/fr/fr/sandales-plates-en-...,Marron |,100% cuir de vache,"35,95 EUR"


In [40]:
# ── HOMME ────────────────────────────────────────────────────────────────
df_homme = scrape_collection(driver, URLS_LISTES["homme"])
df_homme.head()

  118 URLs trouvées
  [1/50] https://www.zara.com/fr/fr/mocassins-en-cuir-avec-semelle-vibram--aaron-levine-x-zara-p12680721.html
  [2/50] https://www.zara.com/fr/fr/ceinture-en-cuir-surpiquee-p06399402.html
  [3/50] https://www.zara.com/fr/fr/blouson-en-jean-a-fermeture-eclair-p03991489.html
  [4/50] https://www.zara.com/fr/fr/mocassins-en-cuir-avec-masque-p12678720.html
  [5/50] https://www.zara.com/fr/fr/bandana-a-imprime-geometrique-graphique-p03921414.html
  [6/50] https://www.zara.com/fr/fr/polo-pique-regular-fit-p00526302.html
  [7/50] https://www.zara.com/fr/fr/pantalon-regular-fit-de-smoking-a-bandes-p04617350.html
  [8/50] https://www.zara.com/fr/fr/lunettes-de-soleil-metalliques-carrees-p02907302.html
  [9/50] https://www.zara.com/fr/fr/chemise-a-carreaux-lin-et-coton-p05588847.html
  [10/50] https://www.zara.com/fr/fr/short-lave-a-plis-p01063428.html
  [11/50] https://www.zara.com/fr/fr/pantalon-regular-fit-plis-smoking-p04415978.html
  [12/50] https://www.zara.com/fr/fr/pa

,nom,url,couleur,matiere,prix
0,MOCASSINS EN CUIR AVEC SEMELLE VIBRAM® AARON L...,https://www.zara.com/fr/fr/mocassins-en-cuir-a...,Bronze |,100% cuir de vache,"99,95 EUR"
1,CEINTURE EN CUIR SURPIQUÉE,https://www.zara.com/fr/fr/ceinture-en-cuir-su...,Noir |,100% cuir de vachette,"35,95 EUR"
2,VESTE EN JEAN À FERMETURE ÉCLAIR,https://www.zara.com/fr/fr/blouson-en-jean-a-f...,Bleu |,100% coton,"59,95 EUR"
3,MOCASSINS EN CUIR AVEC MASQUE,https://www.zara.com/fr/fr/mocassins-en-cuir-a...,Noir |,100% cuir de vache,"59,95 EUR"
4,BANDANA À IMPRIMÉ GÉOMÉTRIQUE GRAPHIQUE,https://www.zara.com/fr/fr/bandana-a-imprime-g...,Bleu marine |,100% coton,"15,95 EUR"


In [41]:
# ── ENFANTS ──────────────────────────────────────────────────────────────
df_fille = scrape_collection(driver, URLS_LISTES["fille"])
df_fille.head()

  147 URLs trouvées
  [1/50] https://www.zara.com/fr/fr/chemise-a-rayures-penn---p02675621.html
  [2/50] https://www.zara.com/fr/fr/bermuda-penn---p05871691.html
  [3/50] https://www.zara.com/fr/fr/robe-drapee-avec-lin-p01594066.html
  [4/50] https://www.zara.com/fr/fr/sac-a-dos-technique-p11134730.html
  [5/50] https://www.zara.com/fr/fr/6-14-ans--short-de-bain-penn---p03366655.html
  [6/50] https://www.zara.com/fr/fr/robe-liseres-penn---p09000003.html
  [7/50] https://www.zara.com/fr/fr/pantalon-large-schiffly-p04786780.html
  [8/50] https://www.zara.com/fr/fr/ballerines-sport-p12430730.html
  [9/50] https://www.zara.com/fr/fr/haut-detail-maille-crochet-p05580605.html
  [10/50] https://www.zara.com/fr/fr/6-14-ans--maillot-de-bain-liseres-penn---p03366604.html
  [11/50] https://www.zara.com/fr/fr/pantalon-wide-leg-en-molleton-penn---p09000002.html
  [12/50] https://www.zara.com/fr/fr/blouson-leger-deperlant-penn---p05644954.html
  [13/50] https://www.zara.com/fr/fr/sac-bandouliere-tec

,nom,url,couleur,matiere,prix
0,CHEMISE À RAYURES PENN ©,https://www.zara.com/fr/fr/chemise-a-rayures-p...,Vert / Écru |,100% coton,"29,95 EUR"
1,BERMUDA PENN ©,https://www.zara.com/fr/fr/bermuda-penn---p058...,Écru |,100% coton,"17,95 EUR"
2,ROBE DRAPÉE EN LIN,https://www.zara.com/fr/fr/robe-drapee-avec-li...,Rose fluo |,61% viscose,"27,95 EUR"
3,SAC À DOS TECHNIQUE,https://www.zara.com/fr/fr/sac-a-dos-technique...,Multicolore |,100% polyamide,"35,95 EUR"
4,6-14 ANS/ SHORT DE BAIN PENN ©,https://www.zara.com/fr/fr/6-14-ans--short-de-...,Bleu |,100% polyamide,"19,95 EUR"


In [42]:
df_garcon = scrape_collection(driver, URLS_LISTES["garçon"])
df_garcon.head()

  169 URLs trouvées
  [1/50] https://www.zara.com/fr/fr/cape-deguisement-kpop-demon-hunters--netflix---p00653748.html
  [2/50] https://www.zara.com/fr/fr/chemise-a-rayures-penn---p02675621.html
  [3/50] https://www.zara.com/fr/fr/bermuda-penn---p05871691.html
  [4/50] https://www.zara.com/fr/fr/porte-bouteille-technique-p11484730.html
  [5/50] https://www.zara.com/fr/fr/blouson-water-resistant-et-windproof-p03121764.html
  [6/50] https://www.zara.com/fr/fr/6-14-ans--short-de-bain-uni-p05644466.html
  [7/50] https://www.zara.com/fr/fr/short-elastiques-en-coton-p09959880.html
  [8/50] https://www.zara.com/fr/fr/6-14-ans--short-de-bain-penn---p03366655.html
  [9/50] https://www.zara.com/fr/fr/lot-de-trois-hauts-unis-et-a-rayures-p01887691.html
  [10/50] https://www.zara.com/fr/fr/robe-liseres-penn---p09000003.html
  [11/50] https://www.zara.com/fr/fr/lot-de-trois-hauts-unis-p01887693.html
  [12/50] https://www.zara.com/fr/fr/6-14-ans--maillot-de-bain-liseres-penn---p03366604.html
  [13/50

,nom,url,couleur,matiere,prix
0,CAPE DÉGUISEMENT KPOP DEMON HUNTERS™ NETFLIX ©,https://www.zara.com/fr/fr/cape-deguisement-kp...,Noir |,100% polyester,"35,95 EUR"
1,CHEMISE À RAYURES PENN ©,https://www.zara.com/fr/fr/chemise-a-rayures-p...,Vert / Écru |,100% coton,"29,95 EUR"
2,BERMUDA PENN ©,https://www.zara.com/fr/fr/bermuda-penn---p058...,Écru |,100% coton,"17,95 EUR"
3,PORTE-BOUTEILLE TECHNIQUE,https://www.zara.com/fr/fr/porte-bouteille-tec...,Multicolore |,100% polyester,"17,95 EUR"
4,BLOUSON WATER RESISTANT ET COUPE-VENT,https://www.zara.com/fr/fr/blouson-water-resis...,Jaune |,100% polyester,"35,95 EUR"


In [43]:
# Fermer le navigateur proprement
driver.quit()
print("Navigateur fermé.")

Navigateur fermé.


In [44]:
df_femme["collection"]="femme"
df_homme["collection"]="homme"
df_fille["collection"]="fille"
df_garcon["collection"]="garçon"

In [47]:
df_total = pd.concat([df_femme, df_homme, df_fille,df_garcon], ignore_index=True)
df_total["categorie"]=df_total["nom"].str.split().str[0]
df_total.head(10)

,nom,url,couleur,matiere,prix,collection,categorie
0,JUPE ASYMÉTRIQUE FLUIDE,https://www.zara.com/fr/fr/jupe-asymetrique-fl...,Sable |,76% polyester,"29,95 EUR",femme,JUPE
1,SHORT MINI DENIM TRF ÉTOILES,https://www.zara.com/fr/fr/short-mini-denim-tr...,Bleu |,100% coton,"29,95 EUR",femme,SHORT
2,ROBE MI-LONGUE FLUIDE,https://www.zara.com/fr/fr/robe-midi-fluide-p0...,Mauve poudré |,88% viscose,"25,99 EUR",femme,ROBE
3,CHEMISE FLUIDE À MANCHES COURTES,https://www.zara.com/fr/fr/chemise-fluide-a-ma...,Mauve clair |,83% viscose,"25,95 EUR",femme,CHEMISE
4,SANDALES PLATES EN DAIM À BOUCLE,https://www.zara.com/fr/fr/sandales-plates-en-...,Marron |,100% cuir de vache,"35,95 EUR",femme,SANDALES
5,JEAN TRF LOOSE FOLDED TAILLE BASSE À RAYURES,https://www.zara.com/fr/fr/jean-trf-loose-fold...,unique |,100% coton,"35,95 EUR",femme,JEAN
6,CHEMISE CINTREE FRONCEE,https://www.zara.com/fr/fr/chemise-cintree-fro...,Écru / Noir |,97% coton,"25,95 EUR",femme,CHEMISE
7,SHORT FLUIDE AVEC BOUTONS,https://www.zara.com/fr/fr/short-fluide-a-bout...,Mauve clair |,83% viscose,"22,95 EUR",femme,SHORT
8,PNG,https://www.zara.com/fr/fr/short-en-denim-trf-...,None,None,"29,95 EUR",femme,PNG
9,COMBI-SHORT LYOCELL,https://www.zara.com/fr/fr/combinaison-courte-...,Noir |,100% lyocell,"35,95 EUR",femme,COMBI-SHORT


In [48]:
# Export CSV
df_total.to_csv("zara_data.csv", index=False, encoding="utf-8-sig")
print("Fichier 'zara_data.csv' exporté.")

Fichier 'zara_data.csv' exporté.
